# Transcription — Pathway A: Whisper

For languages supported by OpenAI Whisper: English, Spanish, French, Māori (`mi`), and ~55 others.

**Not sure if your language is supported?** Run the check in `01_record_and_segment.ipynb` (Step 6) before continuing.

All transcription runs locally. No audio leaves your machine.


## 0. Environment Check

```bash
uv sync --extra dev && source .venv/bin/activate
```

In [ ]:
issues = []
for pkg, mod in [("openai-whisper", "whisper"), ("pandas", "pandas"), ("tqdm", "tqdm")]:
    try:
        __import__(mod)
    except Exception as e:
        issues.append(f"{pkg}: {e}")
if issues:
    for i in issues: print(i)
    print("\nRun: uv sync --extra dev")
else:
    print("All dependencies available.")

## 1. Configure

| Model | Speed | Accuracy | VRAM |
|---|---|---|---|
| `tiny` | ~32x realtime | lower | ~1 GB |
| `base` | ~16x realtime | moderate | ~1 GB |
| `small` | ~6x realtime | good | ~2 GB |
| `medium` | ~2x realtime | very good | ~5 GB |
| `large-v3` | ~1x realtime | best | ~10 GB |

In [ ]:
MODEL_SIZE = "base"      # Change to 'large-v3' for final production pass
LANGUAGE   = "en"        # ISO 639-1 code
INPUT_DIR  = "../test_data/wavs_export_audacity/"
OUTPUT_CSV = "../test_data/transcripts.csv"

## 2. Transcribe

In [ ]:
import os, csv
import whisper
from tqdm import tqdm

print(f"Loading Whisper '{MODEL_SIZE}' model...")
model = whisper.load_model(MODEL_SIZE)

wav_files = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".wav"))
results = []

for filename in tqdm(wav_files, desc="Transcribing"):
    file_id  = os.path.splitext(filename)[0]
    filepath = os.path.join(INPUT_DIR, filename)
    result   = model.transcribe(filepath, language=LANGUAGE)
    results.append({"file_id": file_id, "transcript": result["text"].strip()})

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file_id", "transcript"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved to {OUTPUT_CSV}")

Or use the batch script:
```bash
python ../scripts/batch_transcribe.py \
    --input-dir ../test_data/wavs_export_audacity \
    --output-csv ../test_data/transcripts.csv \
    --model base \
    --language en
```

## 3. Review and Correct

Whisper's output is a starting point. Correct the `transcript` column to exactly match what was spoken — especially names, place names, and dialect vocabulary. Set `knowledge_keeper_reviewed = true` in the metadata after a fluent speaker has confirmed each transcript.

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(f"{len(df)} transcripts")
df

## Next Steps

- Augment if dataset is too small: [04_augmentation.ipynb](04_augmentation.ipynb)
- Export final dataset: [05_export_ljspeech.ipynb](05_export_ljspeech.ipynb)